In [1]:
import torch
from torch_geometric.utils import scatter

In [2]:
# The 3 neighbor vectors stacked as a matrix
neighbor_features = torch.tensor(
    [
        [0.0, 4.0, 1.0],  # Node B
        [1.0, 0.0, 5.0],  # Node C
        [5.0, 1.0, 0.0],  # Node D
    ]
)

In [3]:
neighbor_features

tensor([[0., 4., 1.],
        [1., 0., 5.],
        [5., 1., 0.]])

In [4]:
index = torch.tensor([0, 0, 0])

Exactly! You're connecting it to the right lecture. 👍

The **gating matrix** idea was introduced in the **backpropagation** lecture, but it also helps explain why **GELU** is preferred.

### For ReLU

Forward:

[
y = \max(0,x)
]

Backward:

$$
G =
\operatorname{diag}\!\left(
\frac{d}{dx}\mathrm{GELU}(x_1),
\frac{d}{dx}\mathrm{GELU}(x_2),
\ldots,
\frac{d}{dx}\mathrm{GELU}(x_n)
\right)
$$

This derivative is a **diagonal gating matrix**:

[
G=
\operatorname{diag}(g_1,g_2,\ldots,g_n),
]

where

[
g_i=
\begin{cases}
1 & x_i>0\
0 & x_i\le0
\end{cases}
]

Then backprop is

[
\delta_{\text{in}}
==================

G,\delta_{\text{out}}
]

So yes—the gating matrix is the Jacobian of ReLU. It literally **switches gradients on or off**.

---

### For GELU

Now the gate isn't binary.

Instead of

[
g_i\in{0,1},
]

the diagonal entries become

[
g_i=\frac{d}{dx}\text{GELU}(x_i),
]

which are **continuous values**.

For example:

* very negative → gate ≈ 0
* slightly negative → gate ≈ 0.2
* zero → gate ≈ 0.5
* positive → gate ≈ 1

So the gating matrix becomes

[
G=
\operatorname{diag}(0.03,;0.41,;0.72,;0.99,\ldots)
]

instead of

[
G=
\operatorname{diag}(0,;0,;1,;1,\ldots)
]

---

### The important connection

The **gating matrix is not only a backprop concept**.

* **Forward pass:** the activation (ReLU or GELU) decides how much information passes through.
* **Backward pass:** its derivative (the gating matrix) decides how much gradient passes through.

With ReLU:

* forward gate = hard 0/1
* backward gate = hard 0/1

With GELU:

* forward gate = smooth attenuation
* backward gate = smooth attenuation

That's why GELU gives **smoother gradient flow** and is one reason it trains Transformers more effectively.

This is a great connection to remember for 6.7960:

> **Activation functions are gates in the forward pass, and their Jacobians are gates for gradient flow in the backward pass.**


Here are the formulas, from general Gaussian to the standard normal.

$$
\Phi(x)
=
\int_{-\infty}^{x}
\frac{1}{\sqrt{2\pi}}
e^{-t^2/2}\,dt
$$
### 1. Gaussian (Normal) PDF

If

[
X \sim \mathcal{N}(\mu,\sigma^2),
]

then the probability density function is

```markdown
$$
f(x)=
\frac{1}{\sigma\sqrt{2\pi}}
\exp\!\left(
-\frac{(x-\mu)^2}{2\sigma^2}
\right)
$$
```

---

### 2. Gaussian (Normal) CDF

The cumulative distribution function is

```markdown
$$
F(x)
=
P(X\le x)
=
\int_{-\infty}^{x}
\frac{1}{\sigma\sqrt{2\pi}}
\exp\!\left(
-\frac{(t-\mu)^2}{2\sigma^2}
\right)\,dt
$$
```

There is **no elementary closed-form expression** for this integral.

---

### 3. Standard Normal ((\mu=0,\ \sigma=1))

PDF:

```markdown
$$
\phi(x)
=
\frac{1}{\sqrt{2\pi}}
e^{-x^2/2}
$$
```

CDF:

```markdown
$$
\Phi(x)
=
\int_{-\infty}^{x}
\frac{1}{\sqrt{2\pi}}
e^{-t^2/2}\,dt
$$
```

---

### 4. Relation to the error function (erf)

Most software computes the CDF using the error function:

```markdown
$$
\Phi(x)
=
\frac{1}{2}
\left(
1+
\operatorname{erf}
\left(
\frac{x}{\sqrt{2}}
\right)
\right)
$$
```

This is the formula commonly used in implementations of **GELU**.

---

### ⭐ The one to remember for Transformers

```markdown
$$
\mathrm{GELU}(x)
=
x\,\Phi(x)
=
x\cdot
\frac{1}{2}
\left(
1+
\operatorname{erf}
\left(
\frac{x}{\sqrt{2}}
\right)
\right)
$$
```

In practice, many deep learning libraries use a fast approximation to this expression, but conceptually **GELU = (x) multiplied by the CDF of the standard normal distribution**.

$$
\delta_{\text{in}} = G\,\delta_{\text{out}}
$$

$$
g_i = \frac{d}{dx}\,\mathrm{GELU}(x_i)
$$

In [5]:
import torch
import torch.nn.functional as F

# Example: sequence length = 5
seq_len = 5
d = 64

# Query, Key, Value
Q = torch.randn(seq_len, d)
K = torch.randn(seq_len, d)
V = torch.randn(seq_len, d)

# Attention scores
scores = Q @ K.T / (d**0.5)

print(scores.shape)
# torch.Size([5,5])

torch.Size([5, 5])


In [7]:
mask = torch.triu(torch.ones(seq_len, d), diagonal=1)

In [8]:
mask

tensor([[0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1.,

$$scores = Q @ K.transpose(-2, -1)$$
scores /= math.sqrt(head_dim)

scores.masked_fill_(mask, float("-inf"))

attention = torch.softmax(scores, dim=-1)

output = attention @ V

I think you mean causal self-attention (not “casual”). This is one of the most important concepts in Transformers, especially for GPT models.
What is causal self-attention?
In ordinary self-attention, every token can attend to every other token.
For the sentence:
I love deep learning
each word can see all the others.
I      → I, love, deep, learning
love   → I, love, deep, learning
deep   → I, love, deep, learning
learning → I, love, deep, learning
This is called bidirectional attention (used by models like BERT).

⸻

Causal self-attention
In an autoregressive model like GPT, when predicting the next word, the model must not peek into the future.
Suppose we have
I love deep learning
When predicting “deep”, the model should only know
I love
not “learning”.
So we apply a causal mask.
The allowed attention becomes:
Token	Can attend to
I	I
love	I, love
deep	I, love, deep
learning	I, love, deep, learning
Notice each token can only look left, never right.

⸻

Attention matrix
Without masking:
      I  love deep learn
I      ✓   ✓    ✓    ✓
love   ✓   ✓    ✓    ✓
deep   ✓   ✓    ✓    ✓
learn  ✓   ✓    ✓    ✓
With causal masking:
      I  love deep learn
I      ✓   ✗    ✗    ✗
love   ✓   ✓    ✗    ✗
deep   ✓   ✓    ✓    ✗
learn  ✓   ✓    ✓    ✓
Everything above the diagonal is masked.

⸻

Mathematically
The normal attention scores are
$$A=\frac{QK^T}{\sqrt{d_k}}$$
For causal attention we add a mask M:
$$A=\frac{QK^T}{\sqrt{d_k}}+M$$
where
$$M_{ij}=\begin{cases}0, & j\le i\\-\infty, & j>i\end{cases}$$
After applying softmax,
$$\text{softmax}(-\infty)=0$$,
so the model assigns zero attention to future tokens.

⸻

Why is this necessary?
Suppose we’re training on
The cat sat on the mat.
When predicting “sat”, if the model could already see “on the mat”, the task would become trivial—it would be cheating.
Causal masking forces the model to learn the true next-token prediction task.

⸻

Relation to your MIT Transformer lectures
The typical decoder block is:
Input embeddings
      │
      ▼
Masked (Causal) Multi-Head Self-Attention
      │
Residual + LayerNorm
      │
Feed Forward Network (MLP)
      │
Residual + LayerNorm
The word “Masked” and “Causal” refer to the same triangular mask that prevents looking ahead.

⸻

Quick quiz 🎯
Suppose the sequence is:
A B C D E
When computing the representation of C, which tokens is it allowed to attend to in causal self-attention?
A. A, B, C, D, E
B. A, B, C
C. C
D. D, E


Since you’re studying Transformers, here’s a minimal but realistic PyTorch implementation of how video is converted into tokens and fed to a Transformer. This is essentially what VideoMAE, TimeSformer, ViViT, and modern multimodal vision models do (with architectural variations).

import torch
import torch.nn as nn
# ------------------------------------
# Video
# B = batch
# T = frames
# C = channels
# H,W = resolution
# ------------------------------------
video = torch.randn(2, 16, 3, 224, 224)
# (B,T,C,H,W)
patch_size = 16
embed_dim = 768
# ------------------------------------
# Patch Embedding
# ------------------------------------
class PatchEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(
            in_channels=3,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    def forward(self, x):
        # x = (B*T,C,H,W)
        x = self.proj(x)
        # (B*T,768,14,14)
        x = x.flatten(2)
        # (B*T,768,196)
        x = x.transpose(1,2)
        # (B*T,196,768)
        return x
patch_embed = PatchEmbed()
B,T,C,H,W = video.shape
frames = video.reshape(B*T,C,H,W)
tokens = patch_embed(frames)
print(tokens.shape)

Output

torch.Size([32,196,768])

Every frame becomes

196 patches

Each patch is represented by

768 numbers

⸻

Add Time Dimension

Now reshape back.

tokens = tokens.view(B, T, 196, embed_dim)
print(tokens.shape)

Output

(B,16,196,768)

Meaning

Batch
 └──16 Frames
      └──196 Patches
            └──768-D embedding

⸻

Flatten into Video Tokens

Many models simply flatten everything.

video_tokens = tokens.reshape(B, T*196, embed_dim)
print(video_tokens.shape)

Output

(B,3136,768)

because

16 × 196 = 3136 tokens

Now it looks exactly like an NLP sequence.

[token1 token2 token3 ... token3136]

⸻

Feed into Transformer

encoder = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(
        d_model=768,
        nhead=12,
        batch_first=True
    ),
    num_layers=12
)
features = encoder(video_tokens)
print(features.shape)

Output

(B,3136,768)

Each token has attended to every other token (unless a sparse attention pattern is used).

⸻

Attention Across Space and Time

Suppose

Frame 1
Patch 10

is token

10

and

Frame 2
Patch 10

is token

206

The attention matrix can learn

Patch10(Frame1)
        │
        ▼
Patch10(Frame2)
        │
        ▼
Patch10(Frame3)

tracking the same region over time.

It can also relate different regions:

Face(Frame1)
      │
      ▼
Hand(Frame2)
      │
      ▼
Shoulder(Frame3)

This is how the model learns motion and interactions.

⸻

Computational Challenge

Notice the sequence length:

16 frames × 196 patches = 3136 tokens

Self-attention complexity is

O(N^2)

so

3136^2 \approx 9.8 \text{ million}

attention scores per head, per layer.

That’s why modern video Transformers often use optimizations such as:

* TimeSformer: factorized spatial and temporal attention.
* ViViT: separate spatial and temporal encoders.
* VideoMAE: masked video modeling with efficient pretraining.
* Sparse/windowed attention: reduce quadratic complexity.

This code captures the core idea: video is represented as a sequence of spatiotemporal patch tokens, and the Transformer learns relationships across both space and time through self-attention.

Love it. 😄 Since this is 6.7960, I’ll make these conceptual rather than memorization-heavy.

Hard Quiz: CNN → Backprop → GELU → Transformers

Q1. (Gating)

Suppose a ReLU layer receives

x=[-2,\;3,\;-1,\;5]

The upstream gradient is

\delta=[4,\;1,\;7,\;2].

Without doing matrix multiplication, what gradient leaves the ReLU layer?

⸻

Q2. (Reasoning)

If we replace ReLU with GELU, which statement is most correct?

A. Gradients become larger everywhere.

B. Previously dead neurons now receive exactly the same gradient as positive neurons.

C. Small negative activations can still contribute some gradient.

D. GELU removes the need for residual connections.

⸻

Q3. (Attention)

A CNN needs many layers before the top-right bird can influence the bottom-left bird.

A transformer can do this in one self-attention layer.

Why?

One sentence.

⸻

Q4. (Backprop)

Why is the Jacobian of ReLU a diagonal matrix?

Don’t say “because derivatives are independent.” Explain structurally.

⸻

Q5. (Very hard)

Imagine replacing GELU with

f(x)=2x.

Would this activation increase the expressive power of the network?

A. Yes

B. No

Explain in one sentence.

⸻

Q6. (Research-level intuition)

Suppose every activation in a transformer becomes ReLU instead of GELU.

Training still converges.

Why might validation accuracy still be slightly worse?

Don’t mention “papers say so.” Explain mechanistically.

⸻

Q7. (Connection question)

Complete the sentence:

Attention is to tokens what ________ is to neurons.

⸻

Answer all at once. I’ll grade like an MIT TA and point out any reasoning gaps rather than just marking right or wrong.

This is one of the most important equations in Transformers and LLMs. Let’s connect it to probability.

⸻

Autoregressive Conditional Likelihood

Suppose a sentence is

x=(x_1,x_2,\ldots,x_T)

Instead of modeling the whole sentence at once, we use the chain rule of probability:

$$
P(x_1,x_2,\ldots,x_T)
=
\prod_{t=1}^{T}
P(x_t\mid x_1,\ldots,x_{t-1})
$$

This is called the autoregressive factorization.

⸻

Example

Sentence:

The cat sat

The model computes

$$
P(\text{The},\text{cat},\text{sat})
=
P(\text{The})
\cdot
P(\text{cat}\mid\text{The})
\cdot
P(\text{sat}\mid\text{The},\text{cat})
$$

Each token is predicted conditioned on all previous tokens.

⸻

Conditional Likelihood

During training, the sentence is known.

The objective is to maximize

$$
P(x_t\mid x_{<t})
$$

for every position.

Equivalently,

$$
\max_\theta
\prod_{t=1}^{T}
P_\theta(x_t\mid x_{<t})
$$

where

* \theta = model parameters
* x_{<t} = all previous tokens

⸻

Log-Likelihood

Products are inconvenient, so we take logs:

$$
\log P(x)
=
\sum_{t=1}^{T}
\log
P(x_t\mid x_{<t})
$$

Now the objective becomes

$$
\max_\theta
\sum_{t=1}^{T}
\log
P_\theta(x_t\mid x_{<t})
$$

⸻

Cross-Entropy Loss

Deep learning minimizes negative log-likelihood (NLL):

$$
\mathcal L
=
-
\sum_{t=1}^{T}
\log
P_\theta(x_t\mid x_{<t})
$$

This is exactly the cross-entropy loss used to train GPT-style models.

⸻

Why “autoregressive”?

Because the model uses its own previous sequence to predict the next token.

Input:
The
 ↓
Predict:
cat
Input:
The cat
 ↓
Predict:
sat
Input:
The cat sat
 ↓
Predict:
on

Each prediction regresses on (depends on) the previously generated tokens.

⸻

MIT 6.7960 intuition

Think of an autoregressive transformer as repeatedly answering:

“Given everything I’ve seen so far, what is the probability distribution of the next token?”

That is why GPT is called an autoregressive language model:

$$\boxed{
P(x)=\prod_{t=1}^{T}P(x_t\mid x_{<t})
}$$

This single equation connects:

* ✅ Chain rule of probability
* ✅ Teacher forcing (during training, the model conditions on the true previous tokens)
* ✅ Cross-entropy / negative log-likelihood
* ✅ Causal masking in transformers (the model can only attend to previous tokens, matching the conditional probability P(x_t \mid x_{<t})).